# Modèle gravitaire des exportations canadiennes

**Objectif** : Identifier les marchés sous-exploités pour les exportateurs canadiens à l'aide d'un modèle gravitaire estimé par PPML.

**Contexte EDC** : Ce projet réplique le type d'analyse que l'équipe DREAM (Data, Research, Economics, Analytics, and Modelling) d'Exportation et développement Canada réalise pour orienter les stratégies d'exportation.

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
import sys
sys.path.insert(0, '..')
from config import DATA_CLEAN, OUTPUTS

pd.options.display.float_format = '{:,.2f}'.format

## 1. Exploration des données

Panel gravé : 254 pays partenaires, 20 années (2000–2019).

In [ ]:
df = pd.read_parquet(DATA_CLEAN / 'gravity_panel.parquet')
print(f"Panel : {len(df):,} obs, {df['iso3_d'].nunique()} partenaires, {df['year'].nunique()} ann\u00e9es")
print(f"P\u00e9riode : {df['year'].min()}-{df['year'].max()}")
print(f"Flux > 0 : {(df['trade_value'] > 0).sum():,} ({(df['trade_value'] > 0).mean()*100:.0f}%)")
df.describe()

In [ ]:
# Top 15 partenaires commerciaux du Canada (moyenne 2000-2019)
top15 = df.groupby('iso3_d')['trade_value'].mean().nlargest(15)

fig = px.bar(
    x=top15.index, y=top15.values / 1e9,
    labels={'x': 'Pays', 'y': 'Exportations moyennes (milliards USD)'},
    title='Top 15 partenaires commerciaux du Canada (2000-2019)',
    color=top15.values / 1e9,
    color_continuous_scale='Blues',
)
fig.update_layout(showlegend=False, height=400)
fig.show()

In [ ]:
# \u00c9volution des exportations totales du Canada
annual = df.groupby('year')['trade_value'].sum() / 1e9

fig = px.line(
    x=annual.index, y=annual.values,
    labels={'x': 'Ann\u00e9e', 'y': 'Exportations totales (milliards USD)'},
    title='\u00c9volution des exportations canadiennes (2000-2019)',
)
fig.update_traces(line_width=3)
fig.show()

In [ ]:
# Distribution de la distance avec les partenaires
fig = px.histogram(
    df[df['year'] == 2019], x='dist', nbins=50,
    title='Distribution de la distance Canada-partenaires',
    labels={'dist': 'Distance (km)', 'count': 'Nombre de pays'},
)
fig.show()

## 2. Estimation du mod\u00e8le gravitaire

### 2.1 OLS baseline (benchmark)

Le mod\u00e8le OLS en log-lin\u00e9aire sert de benchmark. Il est biais\u00e9 car il exclut les flux z\u00e9ro.

In [ ]:
# Pr\u00e9paration des variables
df['ln_trade'] = np.log(df['trade_value'].clip(lower=1))
df['ln_dist'] = np.log(df['dist'].clip(lower=1))
df['ln_gdp_d'] = np.log(df['gdp_d'].clip(lower=1))
df['ln_pop_d'] = np.log(df['pop_d'].clip(lower=1))

# OLS sur les flux positifs
df_pos = df[df['trade_value'] > 0].dropna(subset=['ln_dist', 'ln_gdp_d', 'ln_pop_d'])

X_vars = ['ln_dist', 'ln_gdp_d', 'ln_pop_d', 'contig', 'comlang_off', 'rta']
X = sm.add_constant(df_pos[X_vars].astype(float))
y = df_pos['ln_trade']

ols = sm.OLS(y, X).fit(cov_type='HC1')
print(ols.summary())

### 2.2 PPML (Santos Silva & Tenreyro, 2006)

Le PPML corrige deux probl\u00e8mes :
1. **Biais de s\u00e9lection** : inclut les flux z\u00e9ro
2. **H\u00e9t\u00e9rosc\u00e9dasticit\u00e9** : estimateurs consistants m\u00eame en pr\u00e9sence d'h\u00e9t\u00e9rosc\u00e9dasticit\u00e9

In [ ]:
# PPML (GLM Poisson)
df_est = df.dropna(subset=['ln_dist', 'ln_gdp_d', 'ln_pop_d']).copy()

X = sm.add_constant(df_est[X_vars].astype(float))
y = df_est['trade_value'].astype(float)

ppml = sm.GLM(y, X, family=sm.families.Poisson()).fit(cov_type='HC1', maxiter=200)
print(ppml.summary())

### 2.3 Comparaison OLS vs PPML

In [ ]:
comparison = pd.DataFrame({
    'OLS': ols.params[X_vars],
    'PPML': ppml.params[X_vars],
})
comparison['OLS_pval'] = ols.pvalues[X_vars]
comparison['PPML_pval'] = ppml.pvalues[X_vars]

# Ajouter les \u00e9toiles de significativit\u00e9
def stars(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.1: return '*'
    return ''

comparison['OLS_sig'] = comparison['OLS_pval'].apply(stars)
comparison['PPML_sig'] = comparison['PPML_pval'].apply(stars)
print(comparison[['OLS', 'OLS_sig', 'PPML', 'PPML_sig']].to_string())

## 3. Potentiel commercial

Le ratio **Pr\u00e9dit / R\u00e9el** identifie les march\u00e9s sous-exploit\u00e9s (ratio > 1) et sur-exploit\u00e9s (ratio < 1).

In [ ]:
potential = pd.read_csv(OUTPUTS / 'trade_potential.csv')

# Carte mondiale
fig = px.choropleth(
    potential, locations='iso3_d', color='gap_usd',
    hover_name='iso3_d',
    hover_data={'trade_actual': ':,.0f', 'trade_predicted': ':,.0f', 'gap_usd': ':,.0f'},
    color_continuous_scale='RdYlGn',
    title='Gap commercial des exportations canadiennes (USD)',
)
fig.update_layout(geo=dict(showframe=False, projection_type='natural earth'), height=500)
fig.show()

In [ ]:
# Scatter r\u00e9el vs pr\u00e9dit
fig = px.scatter(
    potential, x='trade_actual', y='trade_predicted',
    color='classification', hover_name='iso3_d',
    log_x=True, log_y=True,
    color_discrete_map={
        'Sous-exploit\u00e9': '#4CAF50', '\u00c9quilibre': '#FFC107', 'Sur-exploit\u00e9': '#F44336'
    },
    title='Exportations r\u00e9elles vs pr\u00e9dites',
)
max_val = max(potential['trade_actual'].max(), potential['trade_predicted'].max())
fig.add_trace(go.Scatter(x=[1, max_val], y=[1, max_val], mode='lines',
                          line=dict(dash='dash', color='gray'), name='45\u00b0'))
fig.show()

## 4. Analyse sectorielle

In [ ]:
try:
    sectoral = pd.read_csv(OUTPUTS / 'sectoral_potential.csv')
    sec_summary = sectoral.groupby('sector').agg(
        gap_total=('gap', lambda x: x[x > 0].sum()),
        n_opp=('gap', lambda x: (x > 0).sum()),
    ).sort_values('gap_total', ascending=True).reset_index()

    fig = px.bar(
        sec_summary, y='sector', x='gap_total',
        orientation='h', color='gap_total',
        color_continuous_scale='Greens',
        title='Potentiel inexploit\u00e9 par secteur (USD)',
        labels={'gap_total': 'Gap total (USD)', 'sector': ''},
    )
    fig.update_layout(height=400, showlegend=False)
    fig.show()
except FileNotFoundError:
    print('Ex\u00e9cutez python src/sectors.py d\'abord')

## 5. Sc\u00e9narios contrefactuels

In [ ]:
try:
    cf = pd.read_csv(OUTPUTS / 'counterfactual_summary.csv')
    cf['impact_B'] = cf['impact_total_usd'] / 1e9
    cf['color'] = cf['impact_B'].apply(lambda x: 'Gain' if x > 0 else 'Perte')

    fig = px.bar(
        cf, x='scenario', y='impact_B', color='color',
        color_discrete_map={'Gain': '#4CAF50', 'Perte': '#F44336'},
        title='Impact des sc\u00e9narios contrefactuels (milliards USD)',
        labels={'impact_B': 'Impact (milliards USD)', 'scenario': ''},
        text='impact_B',
    )
    fig.update_traces(texttemplate='%{text:+.1f}B', textposition='outside')
    fig.update_layout(height=500, showlegend=False)
    fig.add_hline(y=0, line_dash='dash', line_color='gray')
    fig.show()
except FileNotFoundError:
    print('Ex\u00e9cutez python src/counterfactual.py d\'abord')

## 6. Conclusions

### R\u00e9sultats cl\u00e9s

1. **177 march\u00e9s analys\u00e9s**, dont 116 sous-exploit\u00e9s ($24.4B de potentiel total)
2. **Secteurs prioritaires** : manufacturier de base ($17.7B), agriculture ($10.2B), \u00e9nergie ($6.4B)
3. **ALE les plus prometteurs** : ASEAN (+$3.1B), Inde (+$2.7B), Mercosur (+$2.4B)
4. **Risques** : sanctions Russie (-$1.2B), r\u00e9cession Chine (-$384M)

### Implications pour EDC

- Concentrer les efforts de facilitation sur les march\u00e9s \u00e0 fort gap (Russie, Allemagne, Espagne, Pologne)
- Prioriser les n\u00e9gociations ALE avec l'ASEAN et l'Inde
- Diversifier les secteurs d'exportation au-del\u00e0 de l'\u00e9nergie vers le manufacturier et l'agriculture

---

*Projet portfolio \u2014 Quantitative Analyst-Economist, EDC*